# First MVP RL

Prompt for MDP: "Give me an overview on Markov Decision Processes and how to use them as a basic building block of my Reinforcement Learning Algorithm"


Prompt for first scaffolding pipeline and classes: "Give me a python script that serves as the scaffolding for the reinforcement learning with a markov decision process"

In [2]:
import numpy as np
import random
from collections import defaultdict

# Class Environment

#TODO: Describe characteristics of ENV class, why we need it and how to use it.

In [3]:
CAPACITY = 50  # kWh
MAX_POWER = 22  # kW
TIME_INTERVALS = 8  # 2:00 PM to 4:00 PM in 15-min intervals
ACTION_SPACE = np.linspace(0, MAX_POWER, 12).tolist()  # Discrete power levels (0, 2, 4, ..., 22 kW)

class SmartChargingEnv:
    """
    Discrete event simulation for the EV charging problem.
    """
    def __init__(self, capacity=CAPACITY, max_power=MAX_POWER):
        self.capacity = capacity
        self.max_power = max_power
        
        # Actions
        self.action_space = np.linspace(0, MAX_POWER, 12).tolist()
        self.max_steps = TIME_INTERVALS
        
        # Environment state
        self.current_step = 0
        self.battery_level = 0.0
        
        # Time coefficients for cost (simulating variable electricity pricing)
        # Randomly initialized for scaffolding
        # TODO: Replace with real data if available or a more complex time-varying function
        self.alpha_t = [random.uniform(0.1, 0.5) for _ in range(self.max_steps)]

    def reset(self):
        self.current_step = 0
        # Assume battery starts empty or at a random low value when arriving at 2 p.m.
        self.battery_level = 0.0 
        # TODO: Consider more realistic initial battery levels based on typical usage patterns
        # E.g. Start at 30% capacity
        # self.battery_level = random.uniform(0, self.capacity * 0.3)
        return self._get_state()

    def _get_state(self):
        # Discretize battery level to integer for basic Q-Learning compatibility
        return (self.current_step, int(self.battery_level))

    def step(self, action_idx):
        power_kw = self.action_space[action_idx]
        
        # Calculate new battery level (Power * 0.25 hours)
        energy_added = power_kw * 0.25
        self.battery_level = min(self.capacity, self.battery_level + energy_added)
        
        # Calculate immediate charging cost (Exponential cost function)
        alpha = self.alpha_t[self.current_step]
        # Using a scaled exponent so math.exp doesn't blow up with 22kW
        charging_cost = alpha * np.exp(power_kw / 10.0)
        reward = -charging_cost
        
        self.current_step += 1
        
        # Check termination and apply demand penalty
        done = False
        if self.current_step >= self.max_steps:
            done = True
            # Stochastic demand generation: Normal distribution (mu=30, sigma=5)
            # TODO: Consider more complex demand patterns based on time of day, weather, etc.
            demand = np.random.normal(30, 5)
            
            if self.battery_level < demand:
                # High penalty for running out of energy
                # TODO: Consider a more nuanced penalty based on how much energy is short
                reward -= 5000
                
        return self._get_state(), reward, done

## Class Agent

#TODO: Describe characteristics of agent class, why we need it and how to use it.

In [4]:
LR = 0.1 # Learning rate
GAMMA = 0.99 # Discount factor
EPSILON = 0.1 # Exploration rate

class QLearningAgent:
    def __init__(self, action_space, lr=LR, gamma=GAMMA, epsilon=EPSILON):
        # Using a dictionary for the Q-table since states are tuples: (step, battery)
        self.q_table = defaultdict(lambda: np.zeros(len(action_space)))
        self.action_space = action_space
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon

    # Epsilon-greedy action selection
    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.randint(0, len(self.action_space) - 1)
        return np.argmax(self.q_table[state])

    # Q-learning update
    def learn(self, s, a, r, s_next, done):
        if done:
            td_target = r
        else:
            best_next_action = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_action]
            
        td_error = td_target - self.q_table[s][a]
        self.q_table[s][a] += self.lr * td_error

## Training

#TODO: Describe Training method used

In [5]:
env = SmartChargingEnv()
agent = QLearningAgent(action_space=env.action_space)

episodes = 5000 # Number of training episodes -> TODO: Adjust based on convergence behavior

# Training loop
for i in range(episodes):
    state = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action_idx = agent.choose_action(state)
        next_state, reward, done = env.step(action_idx)
        
        agent.learn(state, action_idx, reward, next_state, done)
        print(f"Episode {i+1}/{episodes}, Step: {state[0]}, Battery: {state[1]} kWh, Action: {env.action_space[action_idx]} kW, Reward: {reward:.2f}")
        
        state = next_state
        total_reward += reward

print(f"Training finished. Q-Table has learned {len(agent.q_table)} state-action mappings.")

Episode 1/5000, Step: 0, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.40
Episode 1/5000, Step: 1, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.23
Episode 1/5000, Step: 2, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.42
Episode 1/5000, Step: 3, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.40
Episode 1/5000, Step: 4, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.11
Episode 1/5000, Step: 5, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.41
Episode 1/5000, Step: 6, Battery: 0 kWh, Action: 0.0 kW, Reward: -0.47
Episode 1/5000, Step: 7, Battery: 0 kWh, Action: 0.0 kW, Reward: -5000.23
Episode 2/5000, Step: 0, Battery: 0 kWh, Action: 2.0 kW, Reward: -0.49
Episode 2/5000, Step: 1, Battery: 0 kWh, Action: 2.0 kW, Reward: -0.29
Episode 2/5000, Step: 2, Battery: 1 kWh, Action: 0.0 kW, Reward: -0.42
Episode 2/5000, Step: 3, Battery: 1 kWh, Action: 0.0 kW, Reward: -0.40
Episode 2/5000, Step: 4, Battery: 1 kWh, Action: 0.0 kW, Reward: -0.11
Episode 2/5000, Step: 5, Battery: 1 kWh, Action: 0.0 kW, Reward: -0.41
Epi

In [6]:
# Evaluation Function - Assessing the learned policy over multiple simulated days
def evaluate_smart_charging(env, agent, test_episodes=1000):
    print(f"\n--- Starting Evaluation over {test_episodes} simulated days ---")
    
    # Turn off exploration (Pure Exploitation)
    agent.epsilon = 0.0 
    
    failures = 0
    total_charging_cost = 0.0
    
    for _ in range(test_episodes):
        state = env.reset()
        done = False
        episode_cost = 0.0
        
        while not done:
            # Always choose the optimal known action
            action_idx = np.argmax(agent.q_table[state])
            next_state, reward, done = env.step(action_idx)
            
            # If the reward is massive negative, it hit the penalty, not just a cost
            if reward > -500: 
                # Cost is the absolute value of the negative reward
                episode_cost += abs(reward) 
            
            if done and reward <= -500:
                failures += 1
                
            state = next_state
            
        # Only add to average cost if the agent didn't fail
        if reward > -500:
            total_charging_cost += episode_cost
            
    # 3. Calculate Final Metrics
    successful_days = test_episodes - failures
    success_rate = (successful_days / test_episodes) * 100
    avg_cost = total_charging_cost / successful_days if successful_days > 0 else 0
    
    print(f"Success Rate (Shift demand met): {success_rate:.2f}%")
    print(f"Failure Rate (Ran out of energy): {(failures/test_episodes)*100:.2f}%")
    if successful_days > 0:
         print(f"Average Charging Cost per day: ${avg_cost:.2f}")
    else:
         print("Agent failed every single day. No average cost calculated.")


# Baseline: Naive Strategy (Always charge at max power)
def calculate_naive_baseline(env, test_episodes=1000):
    print(f"\n--- Calculating Naive Strategy Baseline ({test_episodes} days) ---")
    
    total_naive_cost = 0.0
    failures = 0
    
    # The action index for 22 kW is the last index in the action space
    max_power_action_idx = len(env.action_space) - 1
    
    for _ in range(test_episodes):
        env.reset()
        done = False
        episode_cost = 0.0
        
        while not done:
            # The Naive Agent completely ignores the state and Q-table
            # It just blasts maximum power every single time
            next_state, reward, done = env.step(max_power_action_idx)
            
            if reward > -500: 
                episode_cost += abs(reward)
                
            if done and reward <= -500:
                failures += 1
                
        if reward > -500:
            total_naive_cost += episode_cost

    successful_days = test_episodes - failures
    success_rate = (successful_days / test_episodes) * 100
    avg_cost = total_naive_cost / successful_days if successful_days > 0 else 0
    
    print(f"Naive Success Rate: {success_rate:.2f}%")
    if successful_days > 0:
         print(f"Average Naive Charging Cost per day: ${avg_cost:.2f}")

In [7]:
# Run the evaluation
evaluate_smart_charging(env, agent, test_episodes=1000)

# Run the baseline
calculate_naive_baseline(env, test_episodes=1000)


--- Starting Evaluation over 1000 simulated days ---
Success Rate (Shift demand met): 98.30%
Failure Rate (Ran out of energy): 1.70%
Average Charging Cost per day: $20.07

--- Calculating Naive Strategy Baseline (1000 days) ---
Naive Success Rate: 99.90%
Average Naive Charging Cost per day: $24.10


In [ ]:
# TODO: Consider adding more sophisticated baselines, such as:
# - A rule-based strategy that charges at a moderate power level (e.g., 10 kW) regardless of the state

# TODO: Create fucntion to run multiple simulations with different random seeds to assess 
# the robustness of the learned policy and baseline strategies.

# TODO: Create visualizations of the learned policy, such as heatmaps of the Q-values for different states, 
# or line plots showing the average cost per episode over time during training.

# TODO: Create Deep Q-Network (DQN) agent for comparison, which can handle continuous state spaces and 
# potentially learn more complex policies.